For week one, this notebook uses the already-filtered static Companies House dataset created in '01_companies_house.ipyn'.

The static dataset can tell us:

- Company has outstanding charges
- Company sector
- Company size proxy
- Company age
- Number of mortgage/charge records
- Number of outstanding charges

But the static dataset cannot tell us:

- Company borrowed from Lloyds
- Company borrowed from any other banks
- Exact lender/security holder name, officer change.

To know whether the company borrowed from Lloyds / HSBC / Barclays, we need the lender name from the Companies House Charges API.

Therefore, this notebook has two parts:

1. Static structured dataset only for week one.
2. API for the already-filtered company numbers.

In [25]:
import os
import time
import requests
import numpy as np
import pandas as pd

from pathlib import Path
from requests.auth import HTTPBasicAuth
from dotenv import load_dotenv

In [26]:
load_dotenv()

CH_API_KEY = os.getenv("COMPANIES_HOUSE_API_KEY")

In [27]:
data_path = r"C:\MSC\Project\lloyds-commercial-banking-intelligence-2026\data\processed\filtered_bb_sme_sectors.zip"
df = pd.read_csv(data_path)

C:\conda_temp\ipykernel_28016\2277955209.py:2: DtypeWarning: Columns (3,43,44,45,46,47,48,49,50,51,52) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_path)


In [28]:
df.columns = df.columns.str.strip()

print("Columns in processed file:")
print(df.columns.tolist())

Columns in processed file:
['CompanyName', 'CompanyNumber', 'RegAddress.CareOf', 'RegAddress.POBox', 'RegAddress.AddressLine1', 'RegAddress.AddressLine2', 'RegAddress.PostTown', 'RegAddress.County', 'RegAddress.Country', 'RegAddress.PostCode', 'CompanyCategory', 'CompanyStatus', 'CountryOfOrigin', 'DissolutionDate', 'IncorporationDate', 'Accounts.AccountRefDay', 'Accounts.AccountRefMonth', 'Accounts.NextDueDate', 'Accounts.LastMadeUpDate', 'Accounts.AccountCategory', 'Returns.NextDueDate', 'Returns.LastMadeUpDate', 'Mortgages.NumMortCharges', 'Mortgages.NumMortOutstanding', 'Mortgages.NumMortPartSatisfied', 'Mortgages.NumMortSatisfied', 'SICCode.SicText_1', 'SICCode.SicText_2', 'SICCode.SicText_3', 'SICCode.SicText_4', 'LimitedPartnerships.NumGenPartners', 'LimitedPartnerships.NumLimPartners', 'URI', 'PreviousName_1.CONDATE', 'PreviousName_1.CompanyName', 'PreviousName_2.CONDATE', 'PreviousName_2.CompanyName', 'PreviousName_3.CONDATE', 'PreviousName_3.CompanyName', 'PreviousName_4.COND

In [29]:
df["CompanyNumber"] = df["CompanyNumber"].astype(str).str.zfill(8)

print("Rows:", len(df))
print("Columns:", len(df.columns))

df.head()

Rows: 869043
Columns: 57


,CompanyName,CompanyNumber,RegAddress.CareOf,RegAddress.POBox,RegAddress.AddressLine1,RegAddress.AddressLine2,RegAddress.PostTown,RegAddress.County,RegAddress.Country,RegAddress.PostCode,...,PreviousName_8.CONDATE,PreviousName_8.CompanyName,PreviousName_9.CONDATE,PreviousName_9.CompanyName,PreviousName_10.CONDATE,PreviousName_10.CompanyName,ConfStmtNextDueDate,ConfStmtLastMadeUpDate,sector,segment
0,!NFLECTION ADVISORY LIMITED,15073164,NaN,NaN,74 SANTERS LANE,NaN,POTTERS BAR,HERTFORDSHIRE,ENGLAND,EN6 2DA,...,NaN,NaN,NaN,NaN,NaN,NaN,28/08/2026,14/08/2025,"Technology, legal & professional",BB/SME
1,!NFOGENIE LTD,13522064,NaN,NaN,71-75 SHELTON STREET,NaN,LONDON,GREATER LONDON,UNITED KINGDOM,WC2H 9JQ,...,NaN,NaN,NaN,NaN,NaN,NaN,03/08/2026,20/07/2025,"Technology, legal & professional",BB
2,!NNOV8 LIMITED,11006939,NaN,NaN,OLD BARN FARM,HARTFIELD ROAD,EDENBRIDGE,NaN,ENGLAND,TN8 5NF,...,NaN,NaN,NaN,NaN,NaN,NaN,24/10/2026,10/10/2025,Fast growth & emerging,BB
3,"""A"" CONCEPT LIMITED",02537158,NaN,NaN,BEAUFORT HOUSE,5 MIDDLESEX STREET,LONDON,NaN,NaN,E1 7AA,...,NaN,NaN,NaN,NaN,NaN,NaN,19/09/2026,05/09/2025,"Technology, legal & professional",BB/SME
4,"""BEECHBANK COURT"" MANAGEMENT COMPANY LIMITED",01382560,NaN,NaN,21 BEECHBANK,NaN,NORWICH,NaN,ENGLAND,NR2 2AL,...,NaN,NaN,NaN,NaN,NaN,NaN,09/05/2027,25/04/2026,"Technology, legal & professional",BB


In [30]:
# keeping only useful columns
useful_columns = [
    "CompanyName",
    "CompanyNumber",
    "CompanyStatus",
    "CompanyCategory",
    "RegAddress.PostCode",
    "SICCode.SicText_1",
    "SICCode.SicText_2",
    "SICCode.SicText_3",
    "SICCode.SicText_4",
    "Accounts.AccountCategory",
    "IncorporationDate",
    "Mortgages.NumMortCharges",
    "Mortgages.NumMortOutstanding",
    "Mortgages.NumMortSatisfied",
    "Mortgages.NumMortPartSatisfied",
    "sector",
    "segment"
]

available_columns = [col for col in useful_columns if col in df.columns]

relationship_df = df[available_columns].copy()

relationship_df.head()

,CompanyName,CompanyNumber,CompanyStatus,CompanyCategory,RegAddress.PostCode,SICCode.SicText_1,SICCode.SicText_2,SICCode.SicText_3,SICCode.SicText_4,Accounts.AccountCategory,IncorporationDate,Mortgages.NumMortCharges,Mortgages.NumMortOutstanding,Mortgages.NumMortSatisfied,Mortgages.NumMortPartSatisfied,sector,segment
0,!NFLECTION ADVISORY LIMITED,15073164,Active,Private Limited Company,EN6 2DA,70229 - Management consultancy activities othe...,NaN,NaN,NaN,TOTAL EXEMPTION FULL,15/08/2023,0,0,0,0,"Technology, legal & professional",BB/SME
1,!NFOGENIE LTD,13522064,Active,Private Limited Company,WC2H 9JQ,58290 - Other software publishing,NaN,NaN,NaN,MICRO ENTITY,21/07/2021,0,0,0,0,"Technology, legal & professional",BB
2,!NNOV8 LIMITED,11006939,Active,Private Limited Company,TN8 5NF,62090 - Other information technology service a...,70229 - Management consultancy activities othe...,NaN,NaN,MICRO ENTITY,11/10/2017,0,0,0,0,Fast growth & emerging,BB
3,"""A"" CONCEPT LIMITED",02537158,Active,Private Limited Company,E1 7AA,64209 - Activities of other holding companies ...,73110 - Advertising agencies,NaN,NaN,TOTAL EXEMPTION FULL,05/09/1990,2,0,2,0,"Technology, legal & professional",BB/SME
4,"""BEECHBANK COURT"" MANAGEMENT COMPANY LIMITED",01382560,Active,Private Limited Company,NR2 2AL,68320 - Management of real estate on a fee or ...,74990 - Non-trading company,NaN,NaN,MICRO ENTITY,07/08/1978,0,0,0,0,"Technology, legal & professional",BB


In [31]:
# Creating company age from IncorporationDate.
# This helps identify young/growing businesses vs older established businesses.
# Young company: May need business account, business credit card, accounting tools.
# Established company:May need relationship review, savings, lending review, insurance review.

if "IncorporationDate" in relationship_df.columns:
    relationship_df["IncorporationDate"]=pd.to_datetime(
        relationship_df["IncorporationDate"],
        errors="coerce"
    )
    today = pd.Timestamp.today().normalize()
    
    relationship_df["company_age_years"] = (
    (today - relationship_df["IncorporationDate"]).dt.days / 365.25).round(1)
else:
    relationship_df["company_age_years"] = np.nan
relationship_df[["CompanyName", "CompanyNumber","company_age_years"]].head()


C:\conda_temp\ipykernel_28016\3233996133.py:7: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  relationship_df["IncorporationDate"]=pd.to_datetime(


,CompanyName,CompanyNumber,company_age_years
0,!NFLECTION ADVISORY LIMITED,15073164,2.8
1,!NFOGENIE LTD,13522064,4.9
2,!NNOV8 LIMITED,11006939,8.7
3,"""A"" CONCEPT LIMITED",02537158,35.8
4,"""BEECHBANK COURT"" MANAGEMENT COMPANY LIMITED",01382560,47.8


In [32]:
# Creating charge / mortgage indicators from the static Companies House dataset.
charge_cols = [
    "Mortgages.NumMortCharges",
    "Mortgages.NumMortOutstanding",
    "Mortgages.NumMortSatisfied",
    "Mortgages.NumMortPartSatisfied"
]
for col in charge_cols:
    if col in relationship_df.columns:
        relationship_df[col] = pd.to_numeric(relationship_df[col], errors="coerce")
    else:
        relationship_df[col]=0

relationship_df["has_any_charge"] = relationship_df["Mortgages.NumMortCharges"] > 0

relationship_df["has_outstanding_charges"] = relationship_df["Mortgages.NumMortOutstanding"] > 0

relationship_df["debt_ratio"] = np.where(
    relationship_df["Mortgages.NumMortCharges"] > 0,
    relationship_df["Mortgages.NumMortOutstanding"] / relationship_df["Mortgages.NumMortCharges"],
    0
).round(2)

relationship_df[
    [
        "CompanyName",
        "CompanyNumber",
        "Mortgages.NumMortCharges",
        "Mortgages.NumMortOutstanding",
        "has_outstanding_charges",
        "debt_ratio"
    ]
].head()


,CompanyName,CompanyNumber,Mortgages.NumMortCharges,Mortgages.NumMortOutstanding,has_outstanding_charges,debt_ratio
0,!NFLECTION ADVISORY LIMITED,15073164,0,0,False,0.0
1,!NFOGENIE LTD,13522064,0,0,False,0.0
2,!NNOV8 LIMITED,11006939,0,0,False,0.0
3,"""A"" CONCEPT LIMITED",02537158,2,0,False,0.0
4,"""BEECHBANK COURT"" MANAGEMENT COMPANY LIMITED",01382560,0,0,False,0.0


In [33]:
BASE_URL = "https://api.company-information.service.gov.uk"

def ch_get(path, params=None):
    
    if not CH_API_KEY:
        raise ValueError(
            "No Companies House API key found. "
            "Check .env file contains COMPANIES_HOUSE_API_KEY."
        )

    url = BASE_URL + path

    response = requests.get(
        url,
        params=params,
        auth=HTTPBasicAuth(CH_API_KEY, "")
    )

    if response.status_code == 404:
        return None

    response.raise_for_status()

    return response.json()

In [34]:
# We only call the API for companies with outstanding charges.
# This is because lender/security holder name is relevant only where charges exist.
#Keeping API_LIMIT small for now, increase if needed
API_LIMIT = 1000

api_companies = (
    relationship_df[
        relationship_df["has_outstanding_charges"] == True
    ][["CompanyName", "CompanyNumber"]]
    .drop_duplicates()
    .copy()
)

if API_LIMIT is not None:
    api_companies = api_companies.head(API_LIMIT)

print("Companies selected for Charges API check:", len(api_companies))

api_companies.head()

Companies selected for Charges API check: 1000


,CompanyName,CompanyNumber
8,"""FIRST LEADER (UK) LIMITED""",01941022
9,"""INTOUCH COMMUNICATION SERVICES"" LIMITED",03606467
10,"""P.& B.""ENGINEERING COMPANY LIMITED",00263728
11,"""PURPLE EMPEROR"" LTD",03079224
41,&FRIENDS GLOBAL LTD,13834625


In [35]:
LLOYDS_KEYWORDS = [
    "lloyds bank",         
    "bank of scotland",
    "halifax",
    "scottish widows",
    "mbna",
    "lloyds banking group",
    "hbos"                 
]

def classify_lender_name(lender_name):
    lender_lower = str(lender_name).lower()

    if any(keyword in lender_lower for keyword in LLOYDS_KEYWORDS):
        return "Lloyds-linked lender"

    return "Other lender"

In [36]:
#Fetch charge information for one company using Companies House API.
def fetch_company_charges(company_number):
    data = ch_get(f"/company/{company_number}/charges")

    if data is None:
        return []

    return data.get("items", [])

In [37]:
# This loops through selected companies from relationship_df.
# For each company:
# - calls Companies House Charges API
# - extracts lender/security holder names
# - classifies lender as Lloyds-linked lender or Other lender

api_lender_rows = []

for _, company in api_companies.iterrows():
    company_name = company["CompanyName"]
    company_number = company["CompanyNumber"]

    try:
        charges = fetch_company_charges(company_number)

        if len(charges) == 0:
            api_lender_rows.append({
                "CompanyName": company_name,
                "CompanyNumber": company_number,
                "charge_number": None,
                "charge_status": "No charges returned by API",
                "charge_created_on": None,
                "charge_delivered_on": None,
                "lender_name": "No lender name found",
                "lender_group": "No lender found",
                "api_status": "No charges returned"
            })

        for charge in charges:
            charge_number = charge.get("charge_number", None)
            charge_status = charge.get("status", "Unknown")
            charge_created_on = charge.get("created_on", None)
            charge_delivered_on = charge.get("delivered_on", None)

            #In the Companies House Charges API, persons_entitled refers to the people or organisations entitled to the charge
            persons_entitled = charge.get("persons_entitled", [])

            if len(persons_entitled) == 0:
                api_lender_rows.append({
                    "CompanyName": company_name,
                    "CompanyNumber": company_number,
                    "charge_number": charge_number,
                    "charge_status": charge_status,
                    "charge_created_on": charge_created_on,
                    "charge_delivered_on": charge_delivered_on,
                    "lender_name": "No lender name found in charge",
                    "lender_group": "No lender found",
                    "api_status": "Success"
                })

            else:
                for person in persons_entitled:
                    lender_name = person.get("name", "Unknown lender")
                    lender_group = classify_lender_name(lender_name)

                    api_lender_rows.append({
                        "CompanyName": company_name,
                        "CompanyNumber": company_number,
                        "charge_number": charge_number,
                        "charge_status": charge_status,
                        "charge_created_on": charge_created_on,
                        "charge_delivered_on": charge_delivered_on,
                        "lender_name": lender_name,
                        "lender_group": lender_group,
                        "api_status": "Success"
                    })


    except Exception as e:
        api_lender_rows.append({
            "CompanyName": company_name,
            "CompanyNumber": company_number,
            "charge_number": None,
            "charge_status": "API error",
            "charge_created_on": None,
            "charge_delivered_on": None,
            "lender_name": str(e),
            "lender_group": "API error",
            "api_status": "API error"
        })

api_lenders_df = pd.DataFrame(api_lender_rows)

print("API lender rows:", len(api_lenders_df))

api_lenders_df.head()

API lender rows: 2687


,CompanyName,CompanyNumber,charge_number,charge_status,charge_created_on,charge_delivered_on,lender_name,lender_group,api_status
0,"""FIRST LEADER (UK) LIMITED""",01941022,2.0,outstanding,2010-03-29,2010-03-31,Grenville Nominees No. 1 Limited and Grenville...,Other lender,Success
1,"""FIRST LEADER (UK) LIMITED""",01941022,1.0,outstanding,2000-03-30,2000-04-03,Marble Arch Tower Limited,Other lender,Success
2,"""INTOUCH COMMUNICATION SERVICES"" LIMITED",03606467,3.0,outstanding,2001-12-04,2001-12-15,Grenville Nominees No.1 Limited and Grenville ...,Other lender,Success
3,"""INTOUCH COMMUNICATION SERVICES"" LIMITED",03606467,2.0,outstanding,2000-02-11,2000-02-16,Hsbc Bank PLC,Other lender,Success
4,"""INTOUCH COMMUNICATION SERVICES"" LIMITED",03606467,1.0,outstanding,1999-07-06,1999-07-20,Midland Bank PLC,Other lender,Success


In [38]:
# API can return multiple charges for one company.
# So we summarise to one row per company.
def combine_unique_values(series):
    values = (
        series
        .dropna()
        .astype(str)
        .replace("", np.nan)
        .dropna()
        .unique()
        .tolist()
    )

    return "; ".join(values)

company_lender_summary = (
    api_lenders_df.groupby(["CompanyName", "CompanyNumber"], as_index=False)
    .agg(
        lender_names_api=("lender_name", combine_unique_values),
        lender_groups_api=("lender_group", combine_unique_values),
        charge_statuses_api=("charge_status", combine_unique_values),
        api_status=("api_status", combine_unique_values)
    )
)

company_lender_summary["has_lloyds_linked_lender"] = (company_lender_summary["lender_groups_api"]
    .str.contains("Lloyds-linked lender", case=False, na=False)
)

company_lender_summary["has_other_lender"] = (company_lender_summary["lender_groups_api"]
    .str.contains("Other lender", case=False, na=False)
)

def get_lender_check_status(row):
    if "API error" in str(row["api_status"]):
        return "API error"

    if row["has_lloyds_linked_lender"]:
        return "Lloyds-linked lender found"

    if row["has_other_lender"]:
        return "Only other lender found"

    if "No charges returned" in str(row["api_status"]):
        return "No charges returned by API"

    return "Checked - no clear lender found"

company_lender_summary["lender_check_status"] = company_lender_summary.apply(
    get_lender_check_status,
    axis=1
)

company_lender_summary.head()

,CompanyName,CompanyNumber,lender_names_api,lender_groups_api,charge_statuses_api,api_status,has_lloyds_linked_lender,has_other_lender,lender_check_status
0,"""FIRST LEADER (UK) LIMITED""",01941022,Grenville Nominees No. 1 Limited and Grenville...,Other lender,outstanding,Success,False,True,Only other lender found
1,"""INTOUCH COMMUNICATION SERVICES"" LIMITED",03606467,Grenville Nominees No.1 Limited and Grenville ...,Other lender,outstanding,Success,False,True,Only other lender found
2,"""P.& B.""ENGINEERING COMPANY LIMITED",00263728,National Westminster Bank PLC; Midland Bank PL...,Other lender,outstanding; fully-satisfied,Success,False,True,Only other lender found
3,"""PURPLE EMPEROR"" LTD",03079224,Yorkshire Building Society (Trading as Norwich...,Other lender; Lloyds-linked lender,outstanding; fully-satisfied,Success,True,True,Lloyds-linked lender found
4,&FRIENDS GLOBAL LTD,13834625,Time Invoice Finance Limited,Other lender,outstanding,Success,False,True,Only other lender found


In [39]:
api_cols_to_remove = [
    "lender_names_api",
    "lender_groups_api",
    "charge_statuses_api",
    "api_status",
    "has_lloyds_linked_lender",
    "has_other_lender",
    "lender_check_status"
]

relationship_df = relationship_df.drop(
    columns=[c for c in api_cols_to_remove if c in relationship_df.columns],
    errors="ignore"
)

relationship_df = relationship_df.merge(
    company_lender_summary,
    on=["CompanyName", "CompanyNumber"],
    how="left"
    
)

relationship_df["lender_names_api"] = relationship_df["lender_names_api"].fillna("Not checked by API")
relationship_df["lender_groups_api"] = relationship_df["lender_groups_api"].fillna("Not checked by API")
relationship_df["charge_statuses_api"] = relationship_df["charge_statuses_api"].fillna("Not checked by API")
relationship_df["api_status"] = relationship_df["api_status"].fillna("Not checked by API")
relationship_df["lender_check_status"] = relationship_df["lender_check_status"].fillna("Not checked by API")

relationship_df["has_lloyds_linked_lender"] = (relationship_df["has_lloyds_linked_lender"].fillna(False).astype(bool))

relationship_df["has_other_lender"] = (relationship_df["has_other_lender"].fillna(False).astype(bool))

## Lloyds services 

### Day-to-day banking
Business account / current account — the everyday account for receiving and making payments.
Overdraft — a flexible facility on the current account to cover short-term gaps.
Business credit card — a card for business spending, with a credit limit.
Payment services — tools to take and make payments (card processing, international payments).
Accounting tools — software/integrations that help with bookkeeping and cash-flow tracking.

### Borrowing & lending
Business loan — a lump sum borrowed and repaid over time with interest.
Lending review — checking the client's current borrowing to see if it still fits their needs.
Refinancing — replacing an existing loan (often from another lender) with a new one, ideally on better terms.
Growth lending — funding aimed at helping a fast-growing company expand.
Debt review — looking at the client's existing charges/debts to advise on restructuring.

### Asset & equipment finance
Asset finance — funding to buy equipment, machinery or vehicles, spreading the cost over time.
Hire purchase — buying an asset in instalments, owning it outright at the end.
Leasing — renting an asset (e.g. machinery) for a period instead of buying it.
Asset-based lending — borrowing secured against company assets (stock, equipment, receivables).

### Cash-flow finance
Working capital support — short-term funding to cover day-to-day running costs (staff, suppliers) while waiting to be paid.
Invoice finance — borrowing against unpaid invoices to get cash now instead of waiting for customers to pay.

### Savings
Business savings— an account to hold spare cash and earn interest (e.g. after a funding round).

### Insurance
Insurance review — checking the business has the right cover for its size and sector.
Cyber insurance — cover against losses from cyber attacks or data breaches.
Professional indemnity insurance — cover for professional firms against claims of negligent advice or services.

### Advisory / relationship
Relationship review — a relationship manager reviewing the whole banking relationship to spot unmet needs and deepen it.

In [ ]:
def flag_lloyds_services(row):
    flags = []

    company_sector = str(row.get("sector", "")).lower()
    sic_1 = str(row.get("SICCode.SicText_1", "")).lower()

    age = row.get("company_age_years", np.nan)
    debt_ratio = row.get("debt_ratio", 0)

    has_outstanding_charges = bool(row.get("has_outstanding_charges", False))
    has_lloyds_lender = bool(row.get("has_lloyds_linked_lender", False))
    has_other_lender = bool(row.get("has_other_lender", False))

    lender_names = row.get("lender_names_api", "Not checked by API")
    lender_groups = row.get("lender_groups_api", "Not checked by API")

    # 1. Lloyds-linked lender found
    if has_lloyds_lender:
        flags.append({
            "service_flag": "Lloyds-linked lender/security holder identified",
            "lloyds_service": "Relationship review, working capital, business savings, insurance review",
            "reason": "Companies House Charges API shows a Lloyds-linked lender/security holder. This indicates a public charge relationship, not confirmed customer status.",
            "structured_signal": "API lender/security holder name contains Lloyds / Bank of Scotland / HBOS / Halifax",
            "lender_names_api": lender_names,
            "lender_groups_api": lender_groups
        })

    # 2. Other lender found
    if has_other_lender and not has_lloyds_lender:
        flags.append({
            "service_flag": "Other lender identified",
            "lloyds_service": "Refinancing conversation, business loan, working capital, invoice finance",
            "reason": "Companies House Charges API shows a lender/security holder, but it is not Lloyds-linked.",
            "structured_signal": "API lender name does not match Lloyds keywords",
            "lender_names_api": lender_names,
            "lender_groups_api": lender_groups
        })

    # 3. Static outstanding charge signal
    if has_outstanding_charges:
        flags.append({
            "service_flag": "Outstanding charge / borrowing activity",
            "lloyds_service": "Lending review, refinancing, overdraft, working capital support",
            "reason": "Static Companies House data shows the company has outstanding charges.",
            "structured_signal": "Mortgages.NumMortOutstanding > 0",
            "lender_names_api": lender_names,
            "lender_groups_api": lender_groups
        })

    # 4. High debt ratio signal
    if debt_ratio >= 0.5:
        flags.append({
            "service_flag": "High share of outstanding charges",
            "lloyds_service": "Debt review, working capital support, invoice finance, asset-based lending",
            "reason": "A high share of the company's registered charges are still outstanding.",
            "structured_signal": "debt_ratio >= 0.5",
            "lender_names_api": lender_names,
            "lender_groups_api": lender_groups
        })

    # 5. Manufacturing sector
    if "manufacturing" in company_sector or "manufactur" in sic_1:
        flags.append({
            "service_flag": "Manufacturing finance support",
            "lloyds_service": "Asset finance, hire purchase, leasing, working capital",
            "reason": "Manufacturing firms may need equipment, machinery and cash-flow support.",
            "structured_signal": "sector or SIC indicates manufacturing",
            "lender_names_api": lender_names,
            "lender_groups_api": lender_groups
        })

    # 6. Technology, legal and professional sector
    if (
        "technology" in company_sector
        or "legal" in company_sector
        or "professional" in company_sector
        or "information" in sic_1
        or "software" in sic_1
        or "consult" in sic_1
        or "legal" in sic_1
        or "account" in sic_1
    ):
        flags.append({
            "service_flag": "Technology / legal / professional support",
            "lloyds_service": "Business account review, cyber insurance, professional indemnity insurance",
            "reason": "Technology, legal and professional firms may need banking, insurance and cash-flow support.",
            "structured_signal": "sector or SIC indicates technology/legal/professional services",
            "lender_names_api": lender_names,
            "lender_groups_api": lender_groups
        })

    # 7. Fast growth and emerging sector
    if "fast growth" in company_sector or "emerging" in company_sector:
        flags.append({
            "service_flag": "Fast-growth business support",
            "lloyds_service": "Growth lending, business credit card, business savings, invoice finance",
            "reason": "Fast-growth and emerging-sector companies may need funding and cash-flow support.",
            "structured_signal": "sector = Fast growth & emerging",
            "lender_names_api": lender_names,
            "lender_groups_api": lender_groups
        })

    # 8. Young company
    if pd.notna(age) and age <=3:
        flags.append({
            "service_flag": "Early-stage business support",
            "lloyds_service": "Business account, overdraft, credit card, accounting tools",
            "reason": "Young companies may need basic banking and short-term finance support.",
            "structured_signal": "company_age_years <= 3",
            "lender_names_api": lender_names,
            "lender_groups_api": lender_groups
        })

    # 9. Established company
    if pd.notna(age) and age>= 10:
        flags.append({
            "service_flag": "Established company relationship review",
            "lloyds_service": "Business savings, lending review, insurance review, payment services",
            "reason": "Established companies may benefit from wider banking relationship review, but customer status is not confirmed from public data.",
            "structured_signal": "company_age_years >= 10",
            "lender_names_api": lender_names,
            "lender_groups_api": lender_groups
        })

    # 10. No clear signal
    if len(flags) == 0:
        flags.append({
            "service_flag": "No strong structured relationship signal",
            "lloyds_service": "General relationship review",
            "reason": "No strong static or API lender signal found.",
            "structured_signal": "No major charge, sector, age or lender signal",
            "lender_names_api": lender_names,
            "lender_groups_api": lender_groups
        })

    return flags

In [41]:
all_flags = []

for _, row in relationship_df.iterrows():
    company_flags = flag_lloyds_services(row)

    for flag in company_flags:
        all_flags.append({
            "CompanyName": row.get("CompanyName"),
            "CompanyNumber": row.get("CompanyNumber"),
            "sector": row.get("sector", "Not available"),
            "segment": row.get("segment", "Not available"),
            "Accounts.AccountCategory": row.get("Accounts.AccountCategory", "Not available"),
            "company_age_years": row.get("company_age_years"),
            "Mortgages.NumMortCharges": row.get("Mortgages.NumMortCharges"),
            "Mortgages.NumMortOutstanding": row.get("Mortgages.NumMortOutstanding"),
            "has_any_charge": row.get("has_any_charge"),
            "has_outstanding_charges": row.get("has_outstanding_charges"),
            "debt_ratio": row.get("debt_ratio"),
            "api_status": row.get("api_status"),
            "has_lloyds_linked_lender": row.get("has_lloyds_linked_lender"),
            "has_other_lender": row.get("has_other_lender"),
            "lender_names_api": flag["lender_names_api"],
            "lender_groups_api": flag["lender_groups_api"],
            "service_flag": flag["service_flag"],
            "lloyds_service": flag["lloyds_service"],
            "reason": flag["reason"],
            "structured_signal": flag["structured_signal"],
            "future_unstructured_enrichment": "GDELT / NewsAPI placeholder"
        })

flags_df = pd.DataFrame(all_flags)

print("Rows in final service flag dataset:", len(flags_df))

flags_df.head()

Rows in final service flag dataset: 1655879


,CompanyName,CompanyNumber,sector,segment,Accounts.AccountCategory,company_age_years,Mortgages.NumMortCharges,Mortgages.NumMortOutstanding,has_any_charge,has_outstanding_charges,...,api_status,has_lloyds_linked_lender,has_other_lender,lender_names_api,lender_groups_api,service_flag,lloyds_service,reason,structured_signal,future_unstructured_enrichment
0,!NFLECTION ADVISORY LIMITED,15073164,"Technology, legal & professional",BB/SME,TOTAL EXEMPTION FULL,2.8,0,0,False,False,...,Not checked by API,False,False,Not checked by API,Not checked by API,Technology / legal / professional support,"Business account review, cyber insurance, prof...","Technology, legal and professional firms may n...",sector or SIC indicates technology/legal/profe...,GDELT / NewsAPI placeholder
1,!NFLECTION ADVISORY LIMITED,15073164,"Technology, legal & professional",BB/SME,TOTAL EXEMPTION FULL,2.8,0,0,False,False,...,Not checked by API,False,False,Not checked by API,Not checked by API,Early-stage business support,"Business account, overdraft, credit card, acco...",Young companies may need basic banking and sho...,company_age_years <= 3,GDELT / NewsAPI placeholder
2,!NFOGENIE LTD,13522064,"Technology, legal & professional",BB,MICRO ENTITY,4.9,0,0,False,False,...,Not checked by API,False,False,Not checked by API,Not checked by API,Technology / legal / professional support,"Business account review, cyber insurance, prof...","Technology, legal and professional firms may n...",sector or SIC indicates technology/legal/profe...,GDELT / NewsAPI placeholder
3,!NNOV8 LIMITED,11006939,Fast growth & emerging,BB,MICRO ENTITY,8.7,0,0,False,False,...,Not checked by API,False,False,Not checked by API,Not checked by API,Technology / legal / professional support,"Business account review, cyber insurance, prof...","Technology, legal and professional firms may n...",sector or SIC indicates technology/legal/profe...,GDELT / NewsAPI placeholder
4,!NNOV8 LIMITED,11006939,Fast growth & emerging,BB,MICRO ENTITY,8.7,0,0,False,False,...,Not checked by API,False,False,Not checked by API,Not checked by API,Fast-growth business support,"Growth lending, business credit card, business...",Fast-growth and emerging-sector companies may ...,sector = Fast growth & emerging,GDELT / NewsAPI placeholder


#### REFERENCES

[1]. https://www.lloydsbank.com/products-and-services.html

[2]. https://www.lloydsbankinggroup.com/who-we-are/our-brands/lloyds-bank.html

[3]. https://developer.company-information.service.gov.uk/developer-guidelines/

[4].  https://developer-specs.company-information.service.gov.uk/companies-house-public-data-api/resources/chargelist?v=latest 

[5]. https://www.gov.uk/guidance/register-a-charge-mortgage-for-a-limited-company
